# DKD + TGSC on CIFAR-100

Enable a Kaggle GPU. Enable Internet for cloning/installing, or supply the code and dependencies in advance. Attach CIFAR-100 and teacher weights. All training logic lives in the shared Python runner. FitNet is deferred.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Edit this cell. Paths below are examples; use your attached artifact locations.
REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
PRIVATE_REPO = True  # Add a Kaggle Secret named GITHUB_TOKEN with repository read access.
REVISION = "main"  # Prefer a fixed commit for reproducibility.
REPO_DIR = Path("/kaggle/working/mdistiller")
CONFIG_NAME = "dkd_resnet32x4_to_resnet8x4.json"
GPU = 0  # One GPU per process; other jobs can use another GPU.
SEED = 27
DATA_ROOT = Path("/kaggle/input/cifar100")  # Parent of cifar-100-python/
TEACHER_CHECKPOINT = Path("/kaggle/input/cifar-teachers/resnet32x4_vanilla/ckpt_epoch_240.pth")
CACHE_ROOT = Path("/kaggle/input/tgsc-crop-bank")  # Parent of bank hash folders
CACHE_WRITE_ROOT = Path("/kaggle/working/tgsc_cache")
OUTPUT_DIR = Path("/kaggle/working/tgsc_runs/dkd_res32x4_res8x4_seed27")
RESUME = None  # E.g. Path('/kaggle/input/previous-run/latest.pt')
DOWNLOAD = False


In [ ]:
if not REPO_DIR.exists():
    if "YOUR_ACCOUNT" in REPO_URL:
        raise ValueError("Set REPO_URL to your pushed repository")
    def clone_repository():
        if not PRIVATE_REPO:
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
            return
        import os
        import tempfile
        from kaggle_secrets import UserSecretsClient
        # Keep the token out of URLs, notebook output, and files.
        clone_env = dict(os.environ, TGSC_GITHUB_TOKEN=UserSecretsClient().get_secret("GITHUB_TOKEN"),
                         GIT_TERMINAL_PROMPT="0")
        with tempfile.TemporaryDirectory() as auth_directory:
            askpass = Path(auth_directory) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['TGSC_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            clone_env["GIT_ASKPASS"] = str(askpass)
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], env=clone_env, check=True)
    clone_repository()
    subprocess.run(["git", "checkout", REVISION], cwd=REPO_DIR, check=True)
else:
    wanted = subprocess.check_output(["git", "rev-parse", REVISION], cwd=REPO_DIR, text=True).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    if wanted != actual:
        raise RuntimeError("Existing checkout differs from REVISION; select the intended checkout before running")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-tgsc.txt")], check=True)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
CONFIG = REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME
RUNNER = REPO_DIR / "tools" / "run_tgsc.py"

def run_tgsc(command, *extra):
    args = [sys.executable, "-u", str(RUNNER), command,
            "--config", str(CONFIG), "--gpu", str(GPU), "--seed", str(SEED),
            "--data-root", str(DATA_ROOT), "--cache-root", str(CACHE_ROOT)]
    if command != "evaluate":
        args += ["--teacher-checkpoint", str(TEACHER_CHECKPOINT)]
    if DOWNLOAD:
        args += ["--download"]
    args += [str(value) for value in extra]
    log_dir = CACHE_WRITE_ROOT if command == "prepare" else OUTPUT_DIR
    log_dir.mkdir(parents=True, exist_ok=True)
    with (log_dir / (command + ".log")).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, args)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise


## Check before training

A compatible prepared crop bank is required. Use the preparation notebook if the runner reports missing cache files.


In [ ]:
run_tgsc("check")


## Train or resume

The runner saves each completed epoch. Training uses the existing TGSC curriculum and default augmentation. No new baseline training is performed.


In [ ]:
extra = ["--output-dir", OUTPUT_DIR]
if RESUME is not None:
    extra += ["--resume", RESUME]
run_tgsc("train", *extra)


## Evaluate the exported model

SimKD exports its projector and reused teacher head as well as the backbone. Evaluation does not require teacher-backbone weights. Save notebook outputs to preserve checkpoints/logs for a later session.


In [ ]:
run_tgsc("evaluate", "--export", OUTPUT_DIR / "inference_best.pt")
